# 03 - LLM Generation & Evaluation

This notebook demonstrates:
1. **LLM answer generation** with citations using Gemini
2. **Re-ranking impact** comparison (with vs without cross-encoder)
3. **Evaluation** on test set (eval.jsonl) with comprehensive metrics

## Setup

In [1]:
import os
from pathlib import Path
from pydantic import SecretStr
from dotenv import load_dotenv
from rich import print as pprint

from rag_system import (
    FAISSVectorStore,
    Embedder,
    RAGGenerator,
    LLMConfig,
    load_eval_data,
    evaluate_rag_system,
    print_evaluation_results,
    analyze_result,
    inspect_citations,  # New utility function
)

# Load environment variables from .env file
load_dotenv()

True

## 1. Load Components

Load the vector store and initialize the RAG generator with Gemini.

In [2]:
# Load vector store and embedder
print("Loading vector store...")
vector_store = FAISSVectorStore()
embedder = Embedder()

print(f"✓ Loaded {vector_store.index.ntotal} vectors")
print(f"✓ Embedding dimension: {embedder.dimension}")

Loading vector store...
Loaded index with 619 vectors from faiss_index\vector_index.faiss
Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2...


c:\Development\environments\aida-venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✓ Model loaded (dimension: 384)
✓ Loaded 619 vectors
✓ Embedding dimension: 384


In [3]:
# Set up Gemini API key (get free tier key from: https://aistudio.google.com/app/apikey)
# Loaded automatically from .env file
# Alternative: Set environment variable before running notebook

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found. Create a .env file with:\\n"
        "  GOOGLE_API_KEY=your-api-key-here\\n"
        "Or get one from: https://aistudio.google.com/app/apikey"
    )

print("✓ API key loaded")

✓ API key loaded


In [8]:
# Initialize RAG generator with re-ranking enabled
config = LLMConfig(
    api_key=SecretStr(api_key),
    model_name="gemini-3.1-flash-lite",
    temperature=0.0,  # Deterministic for evaluation
    top_k=7
)

rag = RAGGenerator(
    vector_store=vector_store,
    embedder=embedder,
    config=config,
    use_reranking=True  # Enable cross-encoder re-ranking
)

print("✓ RAG generator initialized")

✓ Loaded gemini-3.1-flash-lite
Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-12-v2...
✓ Cross-encoder loaded
✓ Re-ranker enabled (cross-encoder)
✓ RAG generator initialized


## 2. Example Queries

Test the RAG system with example questions.

In [9]:
# Example 2: More complex question
question = "¿Qué es la Escuela de Gobierno Abierto y cuál es su objetivo?"

response = rag.generate_answer(question)

inspect_citations(response, show_full_text=True)


CITATION INSPECTION

❓ Question:
   ¿Qué es la Escuela de Gobierno Abierto y cuál es su objetivo?

💬 Answer:
   La Escuela de Gobierno Abierto es una línea de actuación transversal que
   funciona como un elemento facilitador del proceso de gobierno abierto
   [Resumen_grupo_motor_041024.pdf, p.1; resumen_grupo-motor240520.pdf, p.3].
   Su objetivo es sensibilizar sobre los valores del gobierno abierto,
   favoreciendo el diseño colaborativo de políticas públicas y preparando a la
   infancia y juventud para una ciudadanía activa y comprometida
   [resumen_grupo_motor110424.pdf, p.20]. Para ello, se prevé la elaboración de
   materiales informativos, como guías y vídeos, destinados a diversos
   colectivos de la población [Resumen_grupo_motor_14032025.pdf, p.4].

📊 Metadata:
   Total chunks retrieved: 7
   Pages cited: 2
   Has citations: True
   Used reranking: True

📚 DETAILED CITATIONS (2 pages)

────────────────────────────────────────────────────────────────────────────────
Citat

In [11]:
# Example 3: Question requiring multiple sources
question = "¿Qué acciones se están realizando en el compromiso de comunicación clara?"

response = rag.generate_answer(question)

inspect_citations(response, show_full_text=True)


CITATION INSPECTION

❓ Question:
   ¿Qué acciones se están realizando en el compromiso de comunicación clara?

💬 Answer:
   Las acciones incluyen la remisión del manual de comunicación clara al grupo
   motor y el inicio de un contrato de soporte a las unidades del ayuntamiento
   durante 2026 [Resumen_9_reunion_marzo2026.pdf, p.2]. Asimismo, se trabaja en
   la aplicación de directrices mediante una planificación coherente, la
   formación de equipos prescriptores y la revisión de documentos como las
   multas de tráfico para garantizar el derecho a entender de la ciudadanía
   [resumen_grupo_motor110424.pdf, p.15-16; Resumen_grupo_motor_14032025.pdf,
   p.1].

📊 Metadata:
   Total chunks retrieved: 7
   Pages cited: 1
   Has citations: True
   Used reranking: True

📚 DETAILED CITATIONS (1 pages)

────────────────────────────────────────────────────────────────────────────────
Citation 1: Resumen_9_reunion_marzo2026.pdf, p.2
───────────────────────────────────────────────────────────

In [12]:
# Example 4: Question about dates and events
question = "¿Cuándo y dónde se celebrará la IX Cumbre Global de la Alianza para el Gobierno Abierto?"

response = rag.generate_answer(question)

inspect_citations(response, show_full_text=True)


CITATION INSPECTION

❓ Question:
   ¿Cuándo y dónde se celebrará la IX Cumbre Global de la Alianza para el Gobierno Abierto?

💬 Answer:
   La IX Cumbre Global de la Alianza para el Gobierno Abierto se llevará a cabo
   en Vitoria-Gasteiz [Resumen_grupo_motor_14032025.pdf, p.5]. El evento está
   programado para celebrarse durante los días 7 al 9 de octubre
   [Resumen_grupo_motor_14032025.pdf, p.5].

📊 Metadata:
   Total chunks retrieved: 2
   Pages cited: 1
   Has citations: True
   Used reranking: True

📚 DETAILED CITATIONS (1 pages)

────────────────────────────────────────────────────────────────────────────────
Citation 1: Resumen_grupo_motor_14032025.pdf, p.5
────────────────────────────────────────────────────────────────────────────────
📄 Document: Resumen_grupo_motor_14032025.pdf
📖 Page: 5
🔢 Number of chunks: 1
🎯 Best Rerank Score: 8.6920
📊 Best Retrieval Score: 0.7759

📝 Chunks from this page:

  Chunk 1 (index=36):
    📊 Retrieval Score: 0.7759 | 🎯 Rerank Score: 8.6920
    

In [13]:
# Example 5: Question about specific commitments
question = "¿Qué es POV Madrid y para qué público está diseñado?"

response = rag.generate_answer(question)

inspect_citations(response, show_full_text=True)


CITATION INSPECTION

❓ Question:
   ¿Qué es POV Madrid y para qué público está diseñado?

💬 Answer:
   POV Madrid es un espacio juvenil de participación digital gestionado por la
   Dirección General de Participación Ciudadana [resumen_grupo_motor110424.pdf,
   p.27]. Este espacio está diseñado específicamente para permitir la
   participación de la juventud, diferenciándose de Decide Madrid, que está
   orientado a la población en general [Resumen_grupo-motor130924.pdf, p.4].

📊 Metadata:
   Total chunks retrieved: 2
   Pages cited: 2
   Has citations: True
   Used reranking: True

📚 DETAILED CITATIONS (2 pages)

────────────────────────────────────────────────────────────────────────────────
Citation 1: resumen_grupo_motor110424.pdf, p.27
────────────────────────────────────────────────────────────────────────────────
📄 Document: resumen_grupo_motor110424.pdf
📖 Page: 27
🔢 Number of chunks: 1
🎯 Best Rerank Score: 2.2059
📊 Best Retrieval Score: 0.6250

📝 Chunks from this page:

  Chun

## 3. Evaluation on Test Set

Load `eval.jsonl` and evaluate the RAG system using the evaluation utilities.

In [14]:
# Load evaluation dataset
eval_data = load_eval_data(Path("eval.jsonl"))

print(f"✓ Loaded {len(eval_data)} evaluation examples\n")

# Show first example
print("Example entry:")
print(f"  Question: {eval_data[0]['question']}")
print(f"  Expected: {eval_data[0]['expected_answer'][:100]}...")
print(f"  Sources: {len(eval_data[0]['source_passages'])}")

✓ Loaded 15 evaluation examples

Example entry:
  Question: ¿Cuándo tuvo lugar la primera reunión del grupo motor para el cuarto plan de gobierno abierto?
  Expected: La primera reunión del grupo motor tuvo lugar el 24 de noviembre de 2023....
  Sources: 1


In [15]:
# Run RAG system on all evaluation questions
print("Running evaluation...\n")
responses = []

for i, item in enumerate(eval_data, 1):
    question = item['question']
    print(f"[{i}/{len(eval_data)}] {question[:60]}...")
    
    # Generate answer
    response = rag.generate_answer(question)
    responses.append(response)

print("\n✓ Evaluation complete")

Running evaluation...

[1/15] ¿Cuándo tuvo lugar la primera reunión del grupo motor para e...
[2/15] ¿Cuántos representantes de la sociedad civil y del Ayuntamie...
[3/15] ¿Cuáles son los cuatro ejes del marco estratégico de gobiern...
[4/15] ¿Qué es la Escuela de Gobierno Abierto y cuál es su objetivo...
[5/15] ¿Cuántas personas participaron en la consulta pública sobre ...
[6/15] ¿Cuándo y dónde se celebrará la IX Cumbre Global de la Alian...
[7/15] ¿Para qué grupo de edad está diseñado POV Madrid?...
[8/15] ¿Cuántas personas participaron en la primera consulta públic...
[9/15] ¿Qué es THIVIC?...
[10/15] ¿Qué herramienta de inteligencia artificial se menciona en r...
[11/15] ¿Qué nombre tiene la primera fase del proceso de elaboración...
[12/15] ¿Qué es el Consejo Social de la Ciudad de Madrid?...
[13/15] ¿Qué áreas del Ayuntamiento están involucradas en el comprom...
[14/15] ¿Qué es Decide Madrid?...
[15/15] ¿Cuándo se remitirá el manual de comunicación clara al grupo...

✓ Evaluati

## 4. Comprehensive Metrics

Compute all evaluation metrics using the evaluation module:
1. **Citation accuracy**: % of responses with valid citations
2. **Source precision**: % of cited documents that match expected sources
3. **Source recall**: % of expected sources that were cited
4. **Answer similarity**: Semantic similarity between generated and expected answers
5. **Average chunks retrieved**: Retrieval effectiveness

In [16]:
# Compute all metrics
metrics = evaluate_rag_system(eval_data, responses, embedder)

# Display results
print_evaluation_results(metrics)

EVALUATION METRICS

1. Citation Accuracy: 93.3%
   (14/15 responses have citations)

2. Source Precision: 76.7%
   (Average % of cited documents that match expected sources)

3. Source Recall: 80.0%
   (Average % of expected sources that were cited)

4. Answer Similarity: 71.8%
   (Semantic similarity between generated and expected answers)

5. Average Chunks Retrieved: 4.5
   (Average number of chunks retrieved per question)



## 5. Detailed Results Inspection

Inspect individual examples to understand quality.

In [18]:
# Show examples in detail
print("\n" + "=" * 80)
print("DETAILED EXAMPLE RESULTS")
print("=" * 80)

for i in range(len(eval_data)):
    print(f"\n{'=' * 80}")
    print(f"Example {i+1}")
    print(f"{'=' * 80}")
    
    item = eval_data[i]
    response = responses[i]
    
    # Use analyze_result utility function
    analyze_result(
        question=item['question'],
        expected_answer=item['expected_answer'],
        generated_answer=response.answer,
        expected_sources=item['source_passages'],
        generated_citations=[{'document': c.document, 'page': c.page} for c in response.citations],
        embedder=embedder
    )


DETAILED EXAMPLE RESULTS

Example 1

📝 Question:
   ¿Cuándo tuvo lugar la primera reunión del grupo motor para el cuarto plan de gobierno abierto?

✅ Expected Answer:
   La primera reunión del grupo motor tuvo lugar el 24 de noviembre de 2023.

🤖 Generated Answer:
   La primera reunión del grupo motor para el diseño del cuarto plan de gobierno abierto del Ayuntamiento de Madrid tuvo lugar el 24 de noviembre de 2023 [ResumenReunionGA_20231124.pdf, p.1]. En dicho encuentro se llevó a cabo la presentación y constitución formal del grupo [ResumenReunionGA_20231124.pdf, p.1].

📚 Citations: 1
   - ResumenReunionGA_20231124.pdf, p.1

📊 Metrics:
   Source Precision: 100% | Source Recall: 100%
   Answer Similarity: 59%

Example 2

📝 Question:
   ¿Cuántos representantes de la sociedad civil y del Ayuntamiento tiene el grupo motor?

✅ Expected Answer:
   El grupo motor cuenta con 10 representantes de la sociedad civil y 10 del Ayuntamiento.

🤖 Generated Answer:
   El grupo motor cuenta con 10 re